In [10]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kennedywanakacha/industrial-robotics-and-automation-dataset")

print("Path to dataset files:", path)


Path to dataset files: /Users/zadyra/.cache/kagglehub/datasets/kennedywanakacha/industrial-robotics-and-automation-dataset/versions/1


In [11]:
pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import pandas as pd
print(os.listdir(path))

  # Load the CSV file
df = pd.read_csv(os.path.join(path, "robotics_data.csv"))
df.head()

['robotics_data.csv']


,Year,Industry,Robots_Adopted,Productivity_Gain,Cost_Savings,Jobs_Displaced,Training_Hours
0,2015,Manufacturing,107,7.86,170.67,293,161
1,2015,Healthcare,484,24.77,120.19,819,239
2,2015,Logistics,263,20.74,152.53,743,69
3,2016,Manufacturing,253,16.99,195.43,366,472
4,2016,Healthcare,445,11.00,81.85,100,299


In [14]:
df.shape

(27, 7)

In [22]:
df

,Year,Industry,Robots_Adopted,Productivity_Gain,Cost_Savings,Jobs_Displaced,Training_Hours
0,2015,Manufacturing,107,7.86,170.67,293,161
1,2015,Healthcare,484,24.77,120.19,819,239
2,2015,Logistics,263,20.74,152.53,743,69
3,2016,Manufacturing,253,16.99,195.43,366,472
4,2016,Healthcare,445,11.00,81.85,100,299
5,2016,Logistics,412,11.72,33.53,826,377
6,2017,Manufacturing,343,8.69,170.90,812,159
7,2017,Healthcare,122,8.49,20.97,760,295
8,2017,Logistics,309,19.78,79.14,676,350
9,2018,Manufacturing,286,15.71,121.30,935,342


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Year               27 non-null     int64  
 1   Industry           27 non-null     object 
 2   Robots_Adopted     27 non-null     int64  
 3   Productivity_Gain  27 non-null     float64
 4   Cost_Savings       27 non-null     float64
 5   Jobs_Displaced     27 non-null     int64  
 6   Training_Hours     27 non-null     int64  
dtypes: float64(2), int64(4), object(1)
memory usage: 1.6+ KB


In [18]:
df.describe()

,Year,Robots_Adopted,Productivity_Gain,Cost_Savings,Jobs_Displaced,Training_Hours
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,2019.000000,304.407407,15.023333,102.732593,519.555556,230.629630
std,2.631174,119.707740,6.191593,63.100194,309.032900,134.884608
min,2015.000000,105.000000,5.910000,11.710000,39.000000,58.000000
25%,2017.000000,225.000000,8.920000,40.755000,243.000000,99.500000
50%,2019.000000,289.000000,15.710000,110.460000,559.000000,239.000000
75%,2021.000000,391.000000,20.800000,161.600000,786.000000,345.000000
max,2023.000000,487.000000,24.770000,197.250000,961.000000,480.000000


In [20]:
df['Training_Hours'].mean() 

np.float64(230.62962962962962)

In [21]:
df['Cost_Savings'].mean()

np.float64(102.73259259259258)

In [17]:
df[df['Training_Hours'] < 100]

#

,Year,Industry,Robots_Adopted,Productivity_Gain,Cost_Savings,Jobs_Displaced,Training_Hours
2,2015,Logistics,263,20.74,152.53,743,69
10,2018,Healthcare,487,8.63,103.38,828,94
13,2019,Healthcare,367,5.97,81.24,746,67
19,2021,Healthcare,486,21.87,135.18,350,95
21,2022,Manufacturing,370,16.59,197.25,961,58
24,2023,Manufacturing,335,7.86,177.63,262,58
26,2023,Logistics,199,13.83,15.58,94,75


In [27]:
df[(df['Training_Hours'] < 100) & (df['Productivity_Gain'] > 50)]

,Year,Industry,Robots_Adopted,Productivity_Gain,Cost_Savings,Jobs_Displaced,Training_Hours


In [28]:
df[(df['Training_Hours'] < 100) & (df['Productivity_Gain'] > 20)]

,Year,Industry,Robots_Adopted,Productivity_Gain,Cost_Savings,Jobs_Displaced,Training_Hours
2,2015,Logistics,263,20.74,152.53,743,69
19,2021,Healthcare,486,21.87,135.18,350,95


In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

path = '/Users/zadyra/.cache/kagglehub/datasets/kennedywanakacha/industrial-robotics-and-automation-dataset/versions/1'
df = pd.read_csv(os.path.join(path, 'robotics_data.csv'))
print(df.shape)
df.head()

## Q1 — Which industry has the HUGE COST SAVINGS?
Aggregate total `Cost_Savings` per industry across all years.

In [48]:
q1 = (
    df.groupby('Industry')['Cost_Savings']
    .sum()
    .reset_index()
    .rename(columns={'Cost_Savings': 'Total_Cost_Savings'})
    .sort_values('Total_Cost_Savings', ascending=False)
)

print(q1.to_string(index=False))
winner_q1 = q1.iloc[0]
print(f"\n=> Most profitable (cost savings): {winner_q1['Industry']}  (${winner_q1['Total_Cost_Savings']:.2f}M)")


     Industry  Total_Cost_Savings
Manufacturing             1188.35
   Healthcare              946.06
    Logistics              639.37

=> Most profitable (cost savings): Manufacturing  ($1188.35M)


## Q2 — Huge Cost Savings + LESS Robots Adopted
We want high savings achieved with *fewer* robots — i.e. higher **efficiency per robot**.

**Score = Total Cost Savings / Total Robots Adopted**  
A higher score means more savings squeezed out of each robot.

In [33]:
q2 = df.groupby('Industry').agg(
    Total_Cost_Savings=('Cost_Savings', 'sum'),
    Total_Robots=('Robots_Adopted', 'sum')
).reset_index()

q2['Savings_Per_Robot'] = q2['Total_Cost_Savings'] / q2['Total_Robots']
q2 = q2.sort_values('Savings_Per_Robot', ascending=False)

print(q2.to_string(index=False))
winner_q2 = q2.iloc[0]
print(f"\n=> Best industry (high savings, few robots): {winner_q2['Industry']}  "
      f"(${winner_q2['Savings_Per_Robot']:.3f}M per robot)")


     Industry  Total_Cost_Savings  Total_Robots  Savings_Per_Robot
Manufacturing             1188.35          2299           0.516899
   Healthcare              946.06          3170           0.298442
    Logistics              639.37          2750           0.232498

=> Best industry (high savings, few robots): Manufacturing  ($0.517M per robot)


## Q3 — Huge Productivity Gain + Huge Cost Savings + LESS Robots Adopted
Three criteria pull in different directions, so we normalize each metric to 0–1 and combine into a **composite score**:

```
Score = norm(Productivity_Gain) + norm(Cost_Savings) + (1 − norm(Robots_Adopted))
```

- `norm(Productivity_Gain)` → higher is better  
- `norm(Cost_Savings)` → higher is better  
- `1 − norm(Robots_Adopted)` → *fewer* robots = higher score

In [57]:
q3 = df.groupby('Industry').agg(
    Avg_Productivity_Gain=('Productivity_Gain', 'mean'),
    Total_Cost_Savings=('Cost_Savings', 'sum'),
    Total_Robots=('Robots_Adopted', 'sum')
).reset_index()

def minmax(col):
    return (col - col.min()) / (col.max() - col.min())

q3['norm_productivity'] = minmax(q3['Avg_Productivity_Gain'])
q3['norm_savings']      = minmax(q3['Total_Cost_Savings'])
q3['norm_few_robots']   = 1 - minmax(q3['Total_Robots'])   # inverted — fewer = better

q3['Composite_Score'] = q3['norm_productivity'] + q3['norm_savings'] + q3['norm_few_robots']
q3 = q3.sort_values('Composite_Score', ascending=False)

display_cols = ['Industry','Avg_Productivity_Gain','Total_Cost_Savings',
                'Total_Robots','Composite_Score']
print(q3[display_cols].to_string(index=False))
winner_q3 = q3.iloc[0]
print(f"\n=> Best overall industry: {winner_q3['Industry']}  (composite score: {winner_q3['Composite_Score']:.3f} / 3.0)")


     Industry  Avg_Productivity_Gain  Total_Cost_Savings  Total_Robots  Composite_Score
Manufacturing              15.191111             1188.35          2299         2.948515
    Logistics              15.220000              639.37          2750         1.482204
   Healthcare              14.658889              946.06          3170         0.558654

=> Best overall industry: Manufacturing  (composite score: 2.949 / 3.0)


## Q4 — Less Training Hours but Good Productivity & Cost Savings
Score = norm(Productivity_Gain) + norm(Cost_Savings) + (1 − norm(Training_Hours))

Fewer training hours is better → we invert it.

In [58]:
q4 = df.groupby('Industry').agg(
    Avg_Training_Hours  =('Training_Hours',   'mean'),
    Avg_Productivity    =('Productivity_Gain', 'mean'),
    Total_Cost_Savings  =('Cost_Savings',      'sum'),
).reset_index()

def minmax(s): return (s - s.min()) / (s.max() - s.min())

q4['norm_productivity'] = minmax(q4['Avg_Productivity'])
q4['norm_savings']      = minmax(q4['Total_Cost_Savings'])
q4['norm_low_training'] = 1 - minmax(q4['Avg_Training_Hours'])  # fewer = better
q4['Score'] = q4['norm_productivity'] + q4['norm_savings'] + q4['norm_low_training']
q4 = q4.sort_values('Score', ascending=False)

print(q4[['Industry','Avg_Training_Hours','Avg_Productivity','Total_Cost_Savings','Score']].to_string(index=False))
w = q4.iloc[0]
print(f"\n=> Best industry (low training, high output): {w['Industry']}  "
      f"(avg {w['Avg_Training_Hours']:.0f} training hrs, score {w['Score']:.3f}/3.0)")

     Industry  Avg_Training_Hours  Avg_Productivity  Total_Cost_Savings    Score
Manufacturing          226.111111         15.191111             1188.35 2.686796
   Healthcare          218.666667         14.658889              946.06 1.558654
    Logistics          247.111111         15.220000              639.37 1.000000

=> Best industry (low training, high output): Manufacturing  (avg 226 training hrs, score 2.687/3.0)


## Q5 — Which Year Was the Most Successful?
Sum Cost_Savings and average Productivity_Gain per year. Then combine with a composite score.

In [59]:
q5 = df.groupby('Year').agg(
    Total_Cost_Savings  =('Cost_Savings',      'sum'),
    Avg_Productivity    =('Productivity_Gain', 'mean'),
    Total_Robots        =('Robots_Adopted',    'sum'),
).reset_index()

q5['norm_savings']      = minmax(q5['Total_Cost_Savings'])
q5['norm_productivity'] = minmax(q5['Avg_Productivity'])
q5['Score'] = q5['norm_savings'] + q5['norm_productivity']
q5 = q5.sort_values('Score', ascending=False)

print(q5.to_string(index=False))
w = q5.iloc[0]
print(f"\n=> Most successful year: {int(w['Year'])}  "
      f"(total savings ${w['Total_Cost_Savings']:.2f}M, avg productivity {w['Avg_Productivity']:.2f}%, score {w['Score']:.3f}/2.0)")

 Year  Total_Cost_Savings  Avg_Productivity  Total_Robots  norm_savings  norm_productivity    Score
 2015              443.39         17.790000           854      1.000000           1.000000 2.000000
 2021              357.39         17.293333           936      0.582200           0.909202 1.491401
 2020              264.83         17.030000           571      0.132530           0.861060 0.993590
 2022              293.70         15.676667           981      0.272785           0.613650 0.886435
 2018              237.55         15.550000          1022      0.000000           0.590494 0.590494
 2023              323.54         12.980000           888      0.417752           0.120658 0.538410
 2016              310.81         13.236667          1110      0.355908           0.167581 0.523488
 2019              271.56         13.333333          1083      0.165225           0.185253 0.350478
 2017              271.01         12.320000           774      0.162553           0.000000 0.162553


## Q6 — Best Industry for Each Year
Within each year, find the industry with the highest combined score (Productivity_Gain + Cost_Savings normalized across all rows).

In [37]:
df2 = df.copy()
df2['norm_productivity'] = minmax(df2['Productivity_Gain'])
df2['norm_savings']      = minmax(df2['Cost_Savings'])
df2['Row_Score'] = df2['norm_productivity'] + df2['norm_savings']

best_per_year = (
    df2.loc[df2.groupby('Year')['Row_Score'].idxmax()]
    [['Year','Industry','Productivity_Gain','Cost_Savings','Row_Score']]
    .sort_values('Year')
)

print(best_per_year.to_string(index=False))
print("\n--- Win count per industry ---")
print(best_per_year['Industry'].value_counts().to_string())

 Year      Industry  Productivity_Gain  Cost_Savings  Row_Score
 2015    Healthcare              24.77        120.19   1.584672
 2016 Manufacturing              16.99        195.43   1.577678
 2017     Logistics              19.78         79.14   1.098845
 2018 Manufacturing              15.71        121.30   1.110273
 2019     Logistics              12.10        175.57   1.211360
 2020    Healthcare               9.42        188.18   1.137224
 2021    Healthcare              21.87        135.18   1.511698
 2022 Manufacturing              16.59        197.25   1.566278
 2023    Healthcare              17.25        130.33   1.240596

--- Win count per industry ---
Industry
Healthcare       4
Manufacturing    3
Logistics        2


## Q7 — Which Industry Displaced the Most / Least Jobs?

In [38]:
q7 = df.groupby('Industry')['Jobs_Displaced'].agg(
    Total='sum',
    Avg='mean',
    Max='max',
    Min='min'
).reset_index().sort_values('Total', ascending=False)

print(q7.to_string(index=False))

most = q7.iloc[0]
least = q7.iloc[-1]
print(f"\n=> Most jobs displaced:  {most['Industry']}   (total {int(most['Total'])}, avg {most['Avg']:.0f}/year)")
print(f"=> Least jobs displaced: {least['Industry']}  (total {int(least['Total'])}, avg {least['Avg']:.0f}/year)")

print("\n--- Year with most displacement per industry ---")
peak = df.loc[df.groupby('Industry')['Jobs_Displaced'].idxmax()][['Industry','Year','Jobs_Displaced']]
print(peak.to_string(index=False))

     Industry  Total        Avg  Max  Min
    Logistics   4748 527.555556  900   39
   Healthcare   4655 517.222222  828  100
Manufacturing   4625 513.888889  961   64

=> Most jobs displaced:  Logistics   (total 4748, avg 528/year)
=> Least jobs displaced: Manufacturing  (total 4625, avg 514/year)

--- Year with most displacement per industry ---
     Industry  Year  Jobs_Displaced
   Healthcare  2018             828
    Logistics  2019             900
Manufacturing  2022             961


# Additional Insights

## 1 — Trend Analysis Over Time (per industry)

In [ ]:
trends = df.pivot(index='Year', columns='Industry', values=['Robots_Adopted','Productivity_Gain','Cost_Savings'])

print('--- Robot Adoption per Year ---')
print(trends['Robots_Adopted'].to_string())

print('\n--- Productivity Gain (%) per Year ---')
print(trends['Productivity_Gain'].to_string())

print('\n--- Cost Savings ($M) per Year ---')
print(trends['Cost_Savings'].to_string())

# Direction: first vs last 3 years average
print('\n--- Trend Direction (avg first 3 years vs avg last 3 years) ---')
for metric in ['Robots_Adopted', 'Productivity_Gain', 'Cost_Savings']:
    early = df[df['Year'] <= 2017].groupby('Industry')[metric].mean()
    late  = df[df['Year'] >= 2021].groupby('Industry')[metric].mean()
    delta = ((late - early) / early * 100).round(1)
    print(f'\n{metric} % change (2015-17 → 2021-23):')
    print(delta.to_string())

## 2 — Efficiency Ratios

In [ ]:
eff = df.copy()
eff['Jobs_Per_Robot']          = eff['Jobs_Displaced']  / eff['Robots_Adopted']
eff['Savings_Per_Job_Lost']    = eff['Cost_Savings']    / eff['Jobs_Displaced']
eff['Productivity_Per_TrainingHr'] = eff['Productivity_Gain'] / eff['Training_Hours']

print('--- Jobs Displaced per Robot (lower = less disruptive) ---')
print(eff.groupby('Industry')['Jobs_Per_Robot'].mean().round(3).to_string())

print('\n--- Cost Savings per Job Displaced ($M / person) ---')
print(eff.groupby('Industry')['Savings_Per_Job_Lost'].mean().round(4).to_string())

print('\n--- Productivity Gain per Training Hour (efficiency of training) ---')
print(eff.groupby('Industry')['Productivity_Per_TrainingHr'].mean().round(5).to_string())

print('\n--- Year-by-year: Jobs per Robot (all industries) ---')
print(eff.pivot(index='Year', columns='Industry', values='Jobs_Per_Robot').round(3).to_string())

## 3 — Correlation Analysis
Values close to +1 = strong positive link, close to -1 = inverse, near 0 = no relationship.

In [ ]:
numeric_cols = ['Robots_Adopted','Productivity_Gain','Cost_Savings','Jobs_Displaced','Training_Hours']

print('--- Full Correlation Matrix ---')
print(df[numeric_cols].corr().round(3).to_string())

print('\n--- Key Relationships ---')
pairs = [
    ('Robots_Adopted',   'Productivity_Gain', 'More robots → more productivity?'),
    ('Robots_Adopted',   'Jobs_Displaced',    'More robots → more job losses?'),
    ('Training_Hours',   'Productivity_Gain', 'More training → more productivity?'),
    ('Cost_Savings',     'Jobs_Displaced',    'Higher savings come with more displacement?'),
    ('Robots_Adopted',   'Cost_Savings',      'More robots → more savings?'),
]
for a, b, label in pairs:
    r = df[a].corr(df[b])
    strength = 'strong' if abs(r)>0.6 else 'moderate' if abs(r)>0.3 else 'weak'
    direction = 'positive' if r>0 else 'negative'
    print(f'{label}')
    print(f'  r = {r:.3f}  ({strength} {direction} correlation)\n')

print('--- Correlation per Industry ---')
for ind in df['Industry'].unique():
    sub = df[df['Industry']==ind]
    r_robots_prod = sub['Robots_Adopted'].corr(sub['Productivity_Gain'])
    r_robots_jobs = sub['Robots_Adopted'].corr(sub['Jobs_Displaced'])
    r_train_prod  = sub['Training_Hours'].corr(sub['Productivity_Gain'])
    print(f'{ind}: robots↔productivity={r_robots_prod:.3f} | robots↔jobs_lost={r_robots_jobs:.3f} | training↔productivity={r_train_prod:.3f}')

## 4 — Year-over-Year Growth Rates (per Industry)

In [ ]:
yoy = df.sort_values(['Industry','Year']).copy()

for metric in ['Robots_Adopted', 'Productivity_Gain', 'Cost_Savings']:
    yoy[f'{metric}_YoY%'] = (
        yoy.groupby('Industry')[metric]
        .pct_change() * 100
    ).round(1)

print('--- Robot Adoption YoY Growth % ---')
print(yoy.pivot(index='Year', columns='Industry', values='Robots_Adopted_YoY%').to_string())

print('\n--- Productivity Gain YoY Growth % ---')
print(yoy.pivot(index='Year', columns='Industry', values='Productivity_Gain_YoY%').to_string())

print('\n--- Cost Savings YoY Growth % ---')
print(yoy.pivot(index='Year', columns='Industry', values='Cost_Savings_YoY%').to_string())

print('\n--- Average Annual Growth Rate (CAGR-style mean of YoY%) ---')
for metric in ['Robots_Adopted', 'Productivity_Gain', 'Cost_Savings']:
    col = f'{metric}_YoY%'
    avg = yoy.groupby('Industry')[col].mean().round(1)
    print(f'{metric}:')
    print(avg.to_string())
    print()

## 5 — Human Cost vs Financial Gain

In [ ]:
print('--- Cumulative Totals (2015–2023) ---')
cumulative = df.groupby('Industry').agg(
    Total_Robots         =('Robots_Adopted',    'sum'),
    Total_Jobs_Displaced =('Jobs_Displaced',    'sum'),
    Total_Cost_Savings   =('Cost_Savings',      'sum'),
    Total_Training_Hours =('Training_Hours',    'sum'),
).reset_index()
cumulative['Savings_Per_Job_Lost_$M'] = (cumulative['Total_Cost_Savings'] / cumulative['Total_Jobs_Displaced']).round(4)
cumulative['Jobs_Lost_Per_Robot']      = (cumulative['Total_Jobs_Displaced'] / cumulative['Total_Robots']).round(3)
print(cumulative.to_string(index=False))

print('\n--- Grand totals across ALL industries ---')
grand = cumulative[['Total_Robots','Total_Jobs_Displaced','Total_Cost_Savings','Total_Training_Hours']].sum()
print(grand.to_string())
print(f'Overall savings per job lost: ${grand["Total_Cost_Savings"]/grand["Total_Jobs_Displaced"]:.4f}M')
print(f'Overall jobs lost per robot:  {grand["Total_Jobs_Displaced"]/grand["Total_Robots"]:.3f}')

print('\n--- Is displacement accelerating or slowing? (first half vs second half) ---')
for ind in df['Industry'].unique():
    sub = df[df['Industry']==ind].sort_values('Year')
    first_half = sub[sub['Year'] <= 2019]['Jobs_Displaced'].mean()
    sec_half   = sub[sub['Year'] >= 2019]['Jobs_Displaced'].mean()
    trend = 'accelerating ▲' if sec_half > first_half else 'slowing ▼'
    print(f'{ind}: avg displaced 2015-19={first_half:.0f}  vs  2019-23={sec_half:.0f}  → {trend}')

## 6 — Best & Worst Year per Industry

In [ ]:
def minmax(s): return (s - s.min()) / (s.max() - s.min())

df3 = df.copy()
df3['norm_productivity'] = minmax(df3['Productivity_Gain'])
df3['norm_savings']      = minmax(df3['Cost_Savings'])
df3['norm_low_robots']   = 1 - minmax(df3['Robots_Adopted'])
df3['Score'] = df3['norm_productivity'] + df3['norm_savings'] + df3['norm_low_robots']

cols = ['Year','Productivity_Gain','Cost_Savings','Robots_Adopted','Jobs_Displaced','Score']

for ind in df['Industry'].unique():
    sub = df3[df3['Industry']==ind].sort_values('Score', ascending=False)
    print(f'=== {ind} ===')
    print('Best year:')
    print(sub[cols].iloc[[0]].to_string(index=False))
    print('Worst year:')
    print(sub[cols].iloc[[-1]].to_string(index=False))
    print()